<a href="https://colab.research.google.com/github/maltasesra-create/ZeroTrust_Genomic_Homomorphic_MVP/blob/main/homomorphic_scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import tenseal as ts
import numpy as np

# 1. SIFIR-GÜVEN KRİPTOGRAFİK BAĞLAM (Sadece Veri Sahibinde / Laboratuvarda)
# CKKS şeması, genetik risk skorları ve biyoinformatik vektörler üzerinde şifreli matematiksel işlem sağlar.
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.generate_galois_keys()
context.global_scale = 2**40

# Gizli anahtarı (secret_key) ayırıyoruz; bulut sunucuya ASLA gönderilmeyecek!
secret_key = context.secret_key()
context.make_context_public()  # Sunucuya gidecek açık (public) bağlam

print("1. Adım Tamam: Sıfır-Güven şifreleme anahtarları oluşturuldu. Gizli anahtar izole edildi.\n")

# 2. HASSAS GENOMİK VERİ (Örnek: 5 kritik gen bölgesindeki mutasyon/ekspresyon yoğunluğu)
# Gerçek veride bu değerler FASTA/VCF dosyasından çıkarılan sayısal vektörlerdir.
raw_genomic_vector = [0.85, 0.12, 0.94, 0.05, 0.78]

# Veriyi laboratuvardan çıkmadan homomorfik olarak şifreliyoruz:
encrypted_genome = ts.ckks_vector(context, raw_genomic_vector)
print("2. Adım Tamam: Genomik veri şifrelendi. Buluta giden veri anlamsız kriptografik gürültüdür:")
print(f"   Şifreli Paket Önizlemesi: {str(encrypted_genome.serialize()[:40])}...\n")

# 3. BULUT SUNUCU TARAFI (Blind Computation / Kör Hesaplama)
# Bulut sunucuda tanımlı hastalık/patojen ağırlık modeli (Sunucu hastanın DNA'sını GÖREMEZ!)
diagnostic_weights = [0.40, 0.10, 0.35, 0.05, 0.10]

# Bulut sunucu, ŞİFRELİ veri ile ağırlık vektörünü şifreyi çözmeden doğrudan çarpar ve toplar:
encrypted_risk_score = encrypted_genome.dot(diagnostic_weights)
print("3. Adım Tamam: Bulut sunucu veriyi HİÇ GÖRMEDEN homomorfik analiz yaptı ve şifreli sonucu döndürdü.\n")

# 4. SONUÇ DOĞRULAMA (Sadece Yetkili Laboratuvar / Veri Sahibi)
# Şifreli sonuç geri geldiğinde yalnızca yerel gizli anahtar (secret_key) ile açılır:
decrypted_score = encrypted_risk_score.decrypt(secret_key)[0]
expected_score = np.dot(raw_genomic_vector, diagnostic_weights)

print("--- SONUÇ RAPORU ---")
print(f"Homomorfik (Şifreli) Hesaplanan Risk Skoru : {decrypted_score:.6f}")
print(f"Klasik (Açık Veri) Referans Skoru          : {expected_score:.6f}")
print(f"Matematiksel Doğruluk Sapması              : {abs(decrypted_score - expected_score):.8f}")

ModuleNotFoundError: No module named 'tenseal'

In [2]:
!pip install tenseal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 21.3 MB/s  0:00:00


In [4]:
# DNA Veri Koruma Projesi - MVP Başlangıç
def dna_verisini_kontrol_et(dna_dizisi):
    print(f"Gelen DNA Dizisi işleniyor: {dna_dizisi}")
    # İleride buraya homomorfik şifreleme kütüphaneleri (TenSEAL vb.) eklenecek.
    return "Veri Güvenli / Şifrelendi"

if __name__ == "__main__":
    ornek_dna = "ATCG-TACG-GGCC-AATT"
    sonuc = dna_verisini_kontrol_et(ornek_dna)
    print(sonuc)


Gelen DNA Dizisi işleniyor: ATCG-TACG-GGCC-AATT
Veri Güvenli / Şifrelendi


In [6]:
import tenseal as ts
import numpy as np

def dna_verisini_kontrol_et(dna_dizisi):
    print(f"Gelen DNA Dizisi işleniyor: {dna_dizisi}")

    # 1. TenSEAL Context Oluşturma (CKKS şeması: Gerçek sayı ve skor hesaplamaları için)
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()

    # 2. DNA Dizisinden Sayısal Vektör Üretme (Örn: G, C sayıları ve toplam uzunluk)
    gc_sayisi = float(dna_dizisi.count('G') + dna_dizisi.count('C'))
    toplam_uzunluk = float(len(dna_dizisi.replace('-', '')))
    vektor = [gc_sayisi, toplam_uzunluk, 1.0]

    # 3. Veriyi Homomorfik Olarak Şifreleme
    sifreli_vektor = ts.ckks_vector(context, vektor)
    print(f"Şifreli Paket Önizlemesi: {sifreli_vektor.serialize()[:40]}...")

    # 4. Şifreli Veri Üzerinde Analiz / Skorlama (Bulut sunucu veriyi görmeden ağırlıkla çarpar)
    # Örnek Risk Katsayıları: [0.05, 0.02, 0.1]
    risk_katsayilari = [0.05, 0.02, 0.1]
    sifreli_skor = sifreli_vektor.dot(risk_katsayilari)

    # 5. Sonucu Çözme (Decryption - Sadece anahtara sahip taraf okuyabilir)
    cozulmus_skor = sifreli_skor.decrypt()[0]

    print(f"--- SONUÇ RAPORU ---")
    print(f"Homomorphic (Şifreli) Hesaplanan Risk Skoru : {cozulmus_skor:.6f}")
    return f"Veri Güvenli / Şifrelendi - Skor: {cozulmus_skor:.6f}"

if __name__ == "__main__":
    ornek_dna = "ATCG-TACG-GGCC-AATT"
    sonuc = dna_verisini_kontrol_et(ornek_dna)
    print(sonuc)

Gelen DNA Dizisi işleniyor: ATCG-TACG-GGCC-AATT
Şifreli Paket Önizlemesi: b'\n\x01\x03\x12\x9c\x9f\x14^\xa1\x10\x04\x03\x02\x00\x00\x9c\x0f\x05\x00\x00\x00\x00\x00(\xb5/\xfd\xa0a\x00\x06\x00\xe4[\x0e\x8e\xfd\xdf\xdar'...
--- SONUÇ RAPORU ---
Homomorphic (Şifreli) Hesaplanan Risk Skoru : 0.820000
Veri Güvenli / Şifrelendi - Skor: 0.820000


In [7]:
# Raporu metin dosyası olarak kaydetme
rapor_metni = """--- SONUÇ RAPORU ---
Proje: ZeroTrust_Genomic_Homomorphic_MVP
Homomorphic (Şifreli) Hesaplanan Risk Skoru: 0.820000
Durum: Veri Güvenli / Şifrelendi
"""

with open("genomik_skor_raporu.txt", "w", encoding="utf-8") as dosya:
    dosya.write(rapor_metni)

print("Rapor başarıyla 'genomik_skor_raporu.txt' olarak kaydedildi!")

Rapor başarıyla 'genomik_skor_raporu.txt' olarak kaydedildi!


In [8]:
from google.colab import files

# Raporu oluştur
rapor_metni = """--- SONUÇ RAPORU ---\nProje: ZeroTrust_Genomic_Homomorphic_MVP\nHomomorphic (Şifreli) Hesaplanan Risk Skoru: 0.820000\nDurum: Veri Güvenli / Şifrelendi\n"""

with open("genomik_skor_raporu.txt", "w", encoding="utf-8") as f:
    f.write(rapor_metni)

# Doğrudan bilgisayarına indirme penceresini aç
files.download("genomik_skor_raporu.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
import tenseal as ts

# Not: Gerçek projelerde 'context' ve şifreli veri (sifreli_vektor) kaydedilir.
# Burada aynı oturum içinde test edebilmen için anahtar ve şifreli nesneyi simüle ediyoruz:

print("🔒 Şifreli veri kasadan alınıyor...")

# Güvenlik nedeniyle oluşturduğumuz aynı context ve şifreli vektör örneği:
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()

# Örnek olarak az önceki veriyi tekrar şifreliyoruz (gerçek projede doğrudan şifreli dosya okunur):
vektor = [2.0, 16.0, 1.0] # GC sayısı, uzunluk, sabit
sifreli_vektor = ts.ckks_vector(context, vektor)
risk_katsayilari = [0.05, 0.02, 0.1]
sifreli_skor = sifreli_vektor.dot(risk_katsayilari)

print("🔑 Yetkili anahtar ile şifreli veri çözülüyor...")
# İşte şifreyi açıp sonucu okuduğumuz yer:
cozulmus_skor = sifreli_skor.decrypt()[0]

print(f"--- 🔓 AÇILAN ŞİFRELİ RAPOR ---")
print(f"Çözülen Risk Skoru: {cozulmus_skor:.6f}")
print(f"Durum: Şifre başarıyla çözüldü, veri okundu!")

🔒 Şifreli veri kasadan alınıyor...
🔑 Yetkili anahtar ile şifreli veri çözülüyor...
--- 🔓 AÇILAN ŞİFRELİ RAPOR ---
Çözülen Risk Skoru: 0.519999
Durum: Şifre başarıyla çözüldü, veri okundu!


In [11]:
import tenseal as ts

# Simülasyon için context ve veri
context = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
context.global_scale = 2**40
context.generate_galois_keys()

# Orijinal DNA dizimiz (Bunu şifrelemiştik)
orijinal_dna = "ATCG-TACG-GGCC-AATT"

# Şifreleme ve skorlama simülasyonu
vektor = [float(orijinal_dna.count('G') + orijinal_dna.count('C')), float(len(orijinal_dna.replace('-', ''))), 1.0]
sifreli_vektor = ts.ckks_vector(context, vektor)
risk_katsayilari = [0.05, 0.02, 0.1]
sifreli_skor = sifreli_vektor.dot(risk_katsayilari)

# Şifreyi çözme ve raporda gösterme
cozulmus_skor = sifreli_skor.decrypt()[0]

print(f"--- 🔓 ŞİFRESİ ÇÖZÜLEN ANALİZ RAPORU ---")
print(f"İşlenen Gen Dizilimi : {orijinal_dna}")
print(f"Çözülen Risk Skoru   : {cozulmus_skor:.6f}")
print(f"Durum                : Şifre başarıyla çözüldü ve doğrulandı!")

--- 🔓 ŞİFRESİ ÇÖZÜLEN ANALİZ RAPORU ---
İşlenen Gen Dizilimi : ATCG-TACG-GGCC-AATT
Çözülen Risk Skoru   : 0.820000
Durum                : Şifre başarıyla çözüldü ve doğrulandı!
